In [ ]:
import subprocess, sys
subprocess.check_call(["uv", "pip", "install", "--system", "valkey>=6.0", "matplotlib>=3.9"])

![ReasoningCacheHook cold vs warm run flow: cold run calls real APIs and stores results in Valkey, warm run serves all three tool results from Valkey cache](diagrams/nb01-reasoning-cache-cold-vs-warm.svg)

In [ ]:
import sys, os, time
sys.path.insert(0, "../02-production-agent/agent_files")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ["EMBEDDING_MODEL_ID"] = "amazon.titan-embed-text-v2:0"
os.environ["DUFFEL_SECRET_ARN"]  = ""

import valkey

vk = valkey.Valkey(host="127.0.0.1", port=6379, ssl=False, decode_responses=False)
vk.ping()

_real = vk.execute_command
def _mock(*a):
    cmd = a[0].upper() if a else ""
    if cmd == "FT.CREATE": return "OK"
    if cmd == "FT.SEARCH": return [0]
    if cmd == "FT._LIST":  return []
    return _real(*a)
vk.execute_command = _mock

NOVA_MODEL_ID   = "us.amazon.nova-pro-v1:0"
CLAUDE_MODEL_ID = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

QUESTION = "What is the best time of year to visit Japan?"

## In-loop reasoning cache — Nova Pro and Claude Haiku

`ReasoningCacheHook` intercepts the Strands agent loop at four points:

| Hook event | What it does |
|------------|-------------|
| `BeforeInvocationEvent` | KNN lookup for a similar past question → inject plan hint (requires FT.*) |
| `BeforeToolCallEvent` | Exact-match lookup `(tool, args)` in Valkey → swap real tool for stub on hit |
| `AfterToolCallEvent` | Store fresh tool result in Valkey with per-tool TTL |
| `AfterInvocationEvent` | Store question → trajectory mapping for future plan hints |

**The tool result cache is model-agnostic.** The cache key is `hash(tool_name + args)` — no model ID in the key.  
This means: a warm run with Claude gets cache hits from a cold run with Nova, and vice versa.

Three runs, same question:
1. **Nova cold** — Valkey empty, all 3 tools hit real APIs, results stored
2. **Nova warm** — same model, tool results served from Valkey
3. **Claude** — different model, same cached tool results

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from tools import geocode_destination, climate_summary, wikipedia_summary
from reasoning_cache import ReasoningCacheHook, ensure_trajectory_index

SYSTEM_PROMPT = (
    "You are a travel research assistant. Use geocode_destination first to get "
    "coordinates, then climate_summary for weather data, and wikipedia_summary "
    "for visa policies. Maximum 4 sentences, plain text, in the user's language."
)

ensure_trajectory_index(vk)

### Run 1 — Nova Pro cold (empty cache)

Valkey is empty. All three tools execute real network calls.  
⏳ Expect 15–45 s.

In [ ]:
hook_nova_cold = ReasoningCacheHook(vk, vk, threshold=0.85, ttl=3600)
agent_nova = Agent(
    model=BedrockModel(model_id=NOVA_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[hook_nova_cold],
)

start = time.time()
result_nova_cold = agent_nova(QUESTION)
nova_cold_ms = int((time.time() - start) * 1000)

usage = result_nova_cold.metrics.accumulated_usage
nova_cold_tokens = usage["totalTokens"]

print(f"⏱️  {nova_cold_ms} ms  —  📊 {nova_cold_tokens} tokens  —  🔄 {result_nova_cold.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={hook_nova_cold.stats.get('tool_cache_hits', 0)}  tool_executions={hook_nova_cold.stats.get('tool_executions', 0)}")
print(f"\n{result_nova_cold}")

### Run 2 — Nova Pro warm (tool results cached)

Same model, same question. `BeforeToolCallEvent` returns cached results — no API calls.

In [ ]:
hook_nova_warm = ReasoningCacheHook(vk, vk, threshold=0.85, ttl=3600)
agent_nova_warm = Agent(
    model=BedrockModel(model_id=NOVA_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[hook_nova_warm],
)

start = time.time()
result_nova_warm = agent_nova_warm(QUESTION)
nova_warm_ms = int((time.time() - start) * 1000)

usage = result_nova_warm.metrics.accumulated_usage
nova_warm_tokens = usage["totalTokens"]

print(f"⏱️  {nova_warm_ms} ms  —  📊 {nova_warm_tokens} tokens  —  🔄 {result_nova_warm.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={hook_nova_warm.stats.get('tool_cache_hits', 0)}  tool_executions={hook_nova_warm.stats.get('tool_executions', 0)}")

### Run 3 — Claude Haiku (reads Nova's cached tool results)

Different model, same question. The cache key is `hash(tool_name + args)` — no model ID.  
Claude gets the same tool cache hits as Nova warm, without ever calling the real APIs.

In [ ]:
hook_claude = ReasoningCacheHook(vk, vk, threshold=0.85, ttl=3600)
agent_claude = Agent(
    model=BedrockModel(model_id=CLAUDE_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[hook_claude],
)

start = time.time()
result_claude = agent_claude(QUESTION)
claude_ms = int((time.time() - start) * 1000)

usage = result_claude.metrics.accumulated_usage
claude_tokens = usage["totalTokens"]

print(f"⏱️  {claude_ms} ms  —  📊 {claude_tokens} tokens  —  🔄 {result_claude.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={hook_claude.stats.get('tool_cache_hits', 0)}  tool_executions={hook_claude.stats.get('tool_executions', 0)}")
print(f"\n{result_claude}")

In [ ]:
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams["figure.facecolor"] = "white"

labels   = ["Nova cold", "Nova warm", "Claude"]
tokens   = [nova_cold_tokens, nova_warm_tokens, claude_tokens]
latency  = [nova_cold_ms,     nova_warm_ms,     claude_ms]
hits     = [
    hook_nova_cold.stats.get("tool_cache_hits", 0),
    hook_nova_warm.stats.get("tool_cache_hits", 0),
    hook_claude.stats.get("tool_cache_hits", 0),
]
colors = ["#FF7043", "#42A5F5", "#7E57C2"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, vals, title, unit in [
    (axes[0], tokens,  "Tokens",         ""),
    (axes[1], latency, "Latency",        " ms"),
    (axes[2], hits,    "Tool cache hits", ""),
]:
    bars = ax.bar(labels, vals, color=colors, width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(vals, default=1)*0.03,
                f"{v:,}{unit}", ha="center", fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, max(vals, default=1) * 1.35)

fig.suptitle("ReasoningCacheHook — Nova vs Claude, same Valkey cache", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"{'Run':<22} {'Tokens':>8} {'Latency':>9} {'ToolHits':>9}")
print("-" * 52)
for label, tok, lat, hit in zip(labels, tokens, latency, hits):
    print(f"{label:<22} {tok:>8,} {lat:>7} ms {hit:>9}")

In [ ]:
# Flush Valkey for a clean restart
vk.flushdb()
print(f"Flushed — keys remaining: {vk.dbsize()}")